# IT 630 Participation: Your First KNN Model

Before you start, choose File → Save a copy in Drive and work in that copy, or your work won't be saved. When you're done, share your copy (Share → Anyone with the link → Viewer) and submit the link in Canvas.

Today you will walk through a small but complete supervised machine-learning workflow. The goal is not to memorize scikit-learn. The goal is to see how a model score becomes a claim you can defend.

Scenario: an advising office can only contact a limited number of students this week. We want to predict which students are likely to need advising outreach, using only facts that would be known early enough to act.

By the end, you will have loaded a dataset, chosen legal features, split data, scaled correctly, fit KNN, compared to a baseline, caught leakage, and turned model scores into a ranked outreach list.

## 1. Load the advising dataset

This dataset is synthetic. It is fake student data designed for this class, not records from real students. Treat it like a real workflow anyway: start by loading the file and inspecting what one row means.

In [ ]:
from pathlib import Path

import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.neighbors import KNeighborsClassifier
from sklearn.metrics import accuracy_score, precision_score, recall_score, confusion_matrix

DATA_URL = "https://blkfin.github.io/it630/activities/2026-11-13-first-knn/IT630-Week8-Advising-Students.csv"
LOCAL_DATA_PATHS = [
    Path("it630/summer2026/data/week-08-l1-advising-students.csv"),
    Path("../data/week-08-l1-advising-students.csv"),
]


def build_synthetic_advising_dataset(seed=31, row_count=420):
    rng = np.random.default_rng(seed)
    n = row_count
    prior_gpa = np.clip(rng.normal(3.0, 0.48, n), 1.6, 4.0)
    credits_completed = np.clip(rng.normal(46, 22, n), 6, 110).round().astype(int)
    missed_logins = np.clip(rng.poisson(np.clip(2.2 + (3.0 - prior_gpa) * 1.2, 0.4, 6.5)), 0, 14)
    late_assignments = np.clip(
        rng.poisson(np.clip(1.1 + (3.1 - prior_gpa) * 0.9 + missed_logins * 0.18, 0.1, 6)),
        0,
        10,
    )
    diagnostic_quiz_pct = np.clip(
        rng.normal(78 - (3.0 - prior_gpa) * 9 - missed_logins * 1.7 - late_assignments * 1.2, 10, n),
        35,
        100,
    ).round(1)
    financial_hold = rng.binomial(
        1,
        np.clip(0.06 + (3.0 - prior_gpa) * 0.05 + late_assignments * 0.015, 0.02, 0.22),
        n,
    )
    first_gen = rng.binomial(1, 0.34, n)
    risk_score = (
        (2.3 - prior_gpa) * 1.35
        + missed_logins * 0.24
        + late_assignments * 0.24
        + (70 - diagnostic_quiz_pct) * 0.035
        + financial_hold * 0.55
        + first_gen * 0.22
        + rng.normal(0, 0.85, n)
        - 0.55
    )
    outreach_probability = 1 / (1 + np.exp(-risk_score))
    needed_outreach = rng.binomial(1, outreach_probability)
    outreach_emails_sent = np.where(
        needed_outreach == 1,
        rng.poisson(2.7, n) + 1,
        rng.binomial(1, 0.08, n),
    )
    missing_outreach_records = (needed_outreach == 1) & (rng.random(n) < 0.08)
    outreach_emails_sent = np.where(missing_outreach_records, 0, outreach_emails_sent)
    return pd.DataFrame({
        "student_id": [f"S{1000 + i}" for i in range(n)],
        "prior_gpa": prior_gpa.round(2),
        "credits_completed": credits_completed,
        "missed_logins": missed_logins,
        "late_assignments": late_assignments,
        "diagnostic_quiz_pct": diagnostic_quiz_pct,
        "financial_hold": financial_hold,
        "first_gen": first_gen,
        "outreach_emails_sent": outreach_emails_sent,
        "needed_outreach": needed_outreach,
    })


def load_students():
    for path in LOCAL_DATA_PATHS:
        if path.exists():
            return pd.read_csv(path), str(path)
    try:
        return pd.read_csv(DATA_URL), DATA_URL
    except Exception as exc:
        print("Could not load the remote CSV, so this notebook generated the same synthetic dataset locally.")
        print("Load error:", type(exc).__name__)
        return build_synthetic_advising_dataset(), "generated fallback"


students, data_source = load_students()
print("data source:", data_source)
print("shape:", students.shape)
students.head()

**Explain:** What is the grain of this table? In plain English, what does one row represent?

## 2. Name the prediction task

A prediction task should fit this sentence:

**For each ____, predict ____, using only ____ known at decision time.**

For this notebook:

**For each student, predict whether they will need advising outreach, using only early-term facts known before the outreach decision.**

In [ ]:
print(students.columns.tolist())
students["needed_outreach"].value_counts(normalize=True).rename("share")

**Explain:** Which column is the label? Is the label known for past students, new students, or both?

## 3. Choose legal features

A feature is legal only if you could know it before deciding whom to contact. One candidate column below is a trap: it sounds useful, but it is created after outreach begins.

In [ ]:
candidate_features = [
    "prior_gpa",
    "credits_completed",
    "missed_logins",
    "late_assignments",
    "diagnostic_quiz_pct",
    "financial_hold",
    "first_gen",
    "outreach_emails_sent",
]

students[candidate_features + ["needed_outreach"]].head()

In [ ]:
# TODO: choose the features you would trust at prediction time.
# Leave out the column that would only be known after outreach has started.
safe_features = [...]

legal_features = [
    "prior_gpa",
    "credits_completed",
    "missed_logins",
    "late_assignments",
    "diagnostic_quiz_pct",
    "financial_hold",
    "first_gen",
]

if safe_features == [...]:
    raise ValueError("Replace [...] with your chosen feature names before continuing.")

if "outreach_emails_sent" in safe_features:
    raise ValueError(
        "outreach_emails_sent is not knowable at decision time \u2014 it is created "
        "after outreach begins. Leave it out of safe_features; you will add it on "
        "purpose in Section 9 to see leakage."
    )

illegal = [f for f in safe_features if f not in legal_features]
if illegal:
    raise ValueError(
        f"These are not legal feature columns: {illegal}. "
        f"Choose only from: {legal_features}"
    )

print("you chose", len(safe_features), "features:", safe_features)


**Explain before continuing:** Which column did you leave out, and why is it not legal at decision time?

## 4. Split the data before fitting anything

A train/test split is an honesty check. We train on one group of past students and test on a different group the model did not see.

In [ ]:
X = students[safe_features]
y = students["needed_outreach"]

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.25,
    random_state=7,
    stratify=y,
)

print("train rows:", len(X_train))
print("test rows:", len(X_test))
print("test label share:")
print(y_test.value_counts(normalize=True).rename("share"))

## 5. Baseline first

Before trusting a model, compare it to a lazy rule. If most students do not need outreach, then "always predict no outreach" is already right much of the time.

In [ ]:
baseline_accuracy = (y_test == 0).mean()
print(f"Baseline accuracy if we always predict no outreach: {baseline_accuracy:.3f}")

**Explain:** Why can a high baseline accuracy still be useless for an advising office?

## 6. Scale features the honest way

KNN uses distance. A feature measured in credits can dominate a feature measured as GPA unless we scale. The scaler must be fit on the training data only, then applied to test data.

In [ ]:
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

pd.DataFrame(X_train_scaled, columns=safe_features).describe().round(2).loc[["mean", "std"]]

**Explain:** Why is `fit_transform` used on training data, but only `transform` on test data?

## 7. Fit your first KNN model

KNN predicts a new student's label from the labels of nearby past students. `n_neighbors=5` means the model looks at the five nearest past students and lets them vote.

In [ ]:
knn = KNeighborsClassifier(n_neighbors=5)
knn.fit(X_train_scaled, y_train)

pred = knn.predict(X_test_scaled)

print(f"Accuracy:  {accuracy_score(y_test, pred):.3f}")
print(f"Precision: {precision_score(y_test, pred):.3f}")
print(f"Recall:    {recall_score(y_test, pred):.3f}")
print("Confusion matrix [[true 0, false 1], [false 0, true 1]]:")
print(confusion_matrix(y_test, pred))

**Explain:** For advising outreach, which mistake is worse: contacting someone who did not need outreach, or missing someone who did? Why?

## 8. Try different values of k

Small k can be jumpy. Large k can smooth away local signal. Run the comparison and look for the tradeoff.

In [ ]:
rows = []
for k in [1, 3, 5, 7, 15]:
    model = KNeighborsClassifier(n_neighbors=k)
    model.fit(X_train_scaled, y_train)
    k_pred = model.predict(X_test_scaled)
    rows.append({
        "k": k,
        "accuracy": accuracy_score(y_test, k_pred),
        "precision": precision_score(y_test, k_pred),
        "recall": recall_score(y_test, k_pred),
    })

pd.DataFrame(rows).round(3)

**Change:** Add one more value of `k` to the list and rerun the cell. Did the model become more aggressive about predicting outreach, less aggressive, or about the same?

## 9. Now make the leakage mistake on purpose

The column `outreach_emails_sent` looks predictive because it is related to the outcome. But it is not known before the outreach decision. Add it anyway and watch what happens.

In [ ]:
leaky_features = safe_features + ["outreach_emails_sent"]

X_leaky = students[leaky_features]
Xl_train, Xl_test, yl_train, yl_test = train_test_split(
    X_leaky,
    y,
    test_size=0.25,
    random_state=7,
    stratify=y,
)

leaky_scaler = StandardScaler()
Xl_train_scaled = leaky_scaler.fit_transform(Xl_train)
Xl_test_scaled = leaky_scaler.transform(Xl_test)

leaky_knn = KNeighborsClassifier(n_neighbors=5)
leaky_knn.fit(Xl_train_scaled, yl_train)
leaky_pred = leaky_knn.predict(Xl_test_scaled)

print(f"Clean KNN accuracy: {accuracy_score(y_test, pred):.3f}")
print(f"Leaky KNN accuracy: {accuracy_score(yl_test, leaky_pred):.3f}")
print(f"Leaky precision:    {precision_score(yl_test, leaky_pred):.3f}")
print(f"Leaky recall:       {recall_score(yl_test, leaky_pred):.3f}")

**Explain:** The leaky model probably looks much better. Why is that bad news instead of good news?

## 10. Turn the honest model into an action list

The advising office does not need a model trophy. It needs a ranked list of students to contact first. KNN can produce an outreach score with `predict_proba`: the share of nearby neighbors voting for outreach.

In [ ]:
# Rebuild a table for the held-out students and attach the honest model's outreach score.
test_students = students.loc[X_test.index, ["student_id"] + safe_features + ["needed_outreach"]].copy()
test_students["outreach_score"] = knn.predict_proba(X_test_scaled)[:, 1]

top_10 = test_students.sort_values("outreach_score", ascending=False).head(10)
top_10[["student_id", "outreach_score", "needed_outreach"]].round(3)

**Explain:** If the advising office can contact only five students today, what rule would you use to decide who gets contacted? What else would you want to check before using this list in the real world?

## 11. Final claim

Write the strongest honest claim this workflow supports. Use this shape:

**Using [features], this KNN model [does/does not] beat the baseline on held-out students, but it should only be used if [condition/check] because [risk].**